In [5]:
!pip install torch torchvision --quiet
!pip install einops --quiet
!pip install transformers --quiet
!pip install pandas 

In [6]:
import sys
import os
import glob
import cv2
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    jaccard_score,
    classification_report
)

from pathlib import Path

import os
import re
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange


In [7]:
dataset_dir = Path('../EWS-Dataset')

train_dir = dataset_dir / 'train'
test_dir = dataset_dir / 'test'
validation_dir = dataset_dir / 'validation'

In [8]:
PIXELS_PER_IMAGE = 5000

FEATURE_MODES = {
    "RGB": ["r", "g", "b"],
    "RGB+HSV": ["r", "g", "b", "h", "s", "v"],
    "RGB+HSV+Texture": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var"],
    "RGB+HSV+Texture+VegIdx": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var", "exg", "ndi", "veg", "exg_blur3", "exg_blur7"],
}

In [9]:
def estimate_green_coverage(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img = image_tensor * std + mean

    R, G, B = img[0], img[1], img[2]

    ExG = 2*G - R - B
    coverage = (ExG > 0.1).float().mean().item()
    return coverage

def get_growth_stage(coverage):
    if coverage < 0.2:
        return 0
    elif coverage < 0.5:
        return 1
    else: 
        return 2

In [10]:
def extract_vegetation_indices(image_pil):
    """
    Extracts vegetation indices as extra channels for the ViT
    These highlight plant vs soil much more clearly than raw RGB
    """
    img = np.array(image_pil).astype(np.float32) / 255.0
    R, G, B = img[:,:,0], img[:,:,1], img[:,:,2]

    eps = 1e-6

    # Excess Green Index — highlights green plants
    ExG = 2*G - R - B

    # Normalized Difference Index
    NDI = (G - R) / (G + R + eps)

    # Visible Atmospherically Resistant Index
    VARI = (G - R) / (G + R - B + eps)
    VARI = np.clip(VARI, -1, 1)

    # Stack as extra channels → (3, H, W)
    indices = np.stack([ExG, NDI, VARI], axis=0)
    return torch.tensor(indices, dtype=torch.float32)

class WheatDataset(Dataset):
    def __init__(self, img_dir, augment=False):
        self.img_dir = Path(img_dir)
        self.images = sorted([
            p for p in self.img_dir.glob('*.png')
            if '_mask' not in p.stem
        ])
        self.augment = augment
        self.resize  = transforms.Resize((448, 448))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )

    def augment_pair(self, image, mask):
        import torchvision.transforms.functional as TF
        import random
        if random.random() > 0.5:
            image, mask = TF.hflip(image), TF.hflip(mask)
        if random.random() > 0.5:
            image, mask = TF.vflip(image), TF.vflip(mask)
        angle = random.choice([0, 90, 180, 270])
        image, mask = TF.rotate(image, angle), TF.rotate(mask, angle)
        if random.random() > 0.5:
            image = TF.adjust_brightness(image, random.uniform(0.7, 1.3))
            image = TF.adjust_contrast(image,   random.uniform(0.7, 1.3))
        return image, mask

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        mask_path = self.img_dir / (img_path.stem + '_mask.png')

        image_pil = Image.open(img_path).convert('RGB')
        mask_pil = Image.open(mask_path).convert('L')

        image_pil = self.resize(image_pil)
        mask_pil = self.resize(mask_pil)

        if self.augment:
            image_pil, mask_pil = self.augment_pair(image_pil, mask_pil)

        rgb = self.normalize(self.to_tensor(image_pil))

        veg = extract_vegetation_indices(image_pil)

        image = torch.cat([rgb, veg], dim=0)
        mask = self.to_tensor(mask_pil)
        mask = (mask < 0.5).float()
        return image, mask
        
train_dataset = WheatDataset('../EWS-Dataset/train',      True)
val_dataset   = WheatDataset('../EWS-Dataset/validation', False)
test_dataset  = WheatDataset('../EWS-Dataset/test',       False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)
test_loader  = DataLoader(test_dataset,  batch_size=8)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
img, mask = train_dataset[0]
print(f"Image shape: {img.shape}")
print(f"Mask shape:  {mask.shape}")
print(f"Mask unique values: {mask.unique()}")

Train: 142 | Val: 24 | Test: 24
Image shape: torch.Size([6, 448, 448])
Mask shape:  torch.Size([1, 448, 448])
Mask unique values: tensor([0., 1.])


In [11]:
from transformers import SegformerForSemanticSegmentation

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=2,                  # 0=background, 1=plant
    ignore_mismatched_sizes=True
)

old_conv = model.segformer.encoder.patch_embeddings[0].proj
new_conv = nn.Conv2d(
    in_channels=6,
    out_channels = old_conv.out_channels,
    kernel_size = old_conv.kernel_size,
    stride = old_conv.stride,
    padding = old_conv.padding
)

with torch.no_grad():
    new_conv.weight[:, :3, :, :] = old_conv.weight  # copy RGB weights
    new_conv.weight[:, 3:, :, :] = 0.01             # small init for new channels
    new_conv.bias.data = old_conv.bias.data

model.segformer.encoder.patch_embeddings[0].proj = new_conv
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f"Running on: {device}")

You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different ta

Running on: cpu


In [12]:
import torch.nn.functional as F

def dice_loss(pred, target, smooth=1):
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def combined_loss(logits, masks):
    logits = F.interpolate(logits, size=masks.shape[-2:],
                           mode = 'bilinear', align_corners=False)
    ce = F.cross_entropy(logits, masks, weight=class_weights)
    pred = logits.softmax(dim=1)[:,1]
    dl = dice_loss(pred, masks.float())
    return ce + dl

def compute_class_weights(dataset, num_samples=50):
    plant_pixels = 0
    bg_pixels    = 0
    for i in range(min(num_samples, len(dataset))):
        _, mask = dataset[i]
        plant_pixels += (mask == 1).sum().item()
        bg_pixels    += (mask == 0).sum().item()
    total   = plant_pixels + bg_pixels
    w_bg    = total / (2 * bg_pixels)
    w_plant = total / (2 * plant_pixels)
    print(f"Background weight: {w_bg:.3f} | Plant weight: {w_plant:.3f}")
    return torch.tensor([w_bg, w_plant], dtype=torch.float32).to(device)

class_weights = compute_class_weights(train_dataset)

Background weight: 0.664 | Plant weight: 2.021


In [15]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

optimizer = AdamW([
    {"params": model.segformer.parameters(),   "lr": 1e-5},  # pretrained ViT encoder
    {"params": model.decode_head.parameters(), "lr": 1e-4},  # fresh decoder
], weight_decay=0.01)

EPOCHS    = 50
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

def compute_iou(pred_mask, true_mask):
    pred_mask = pred_mask.bool()
    true_mask = true_mask.bool()
    intersection = (pred_mask & true_mask).sum().float()
    union = (pred_mask | true_mask).sum().float()

    return (intersection / (union + 1e-6)).item()

best_val_iou = 0.0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_losses = []
    for images, masks in train_loader:
        images = images.to(device)
        masks  = masks.to(device).long().squeeze(1)   # (B, H, W)

        outputs = model(pixel_values=images)
        loss    = combined_loss(outputs.logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    scheduler.step()

    # --- Validate ---
    model.eval()
    val_ious = []
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks  = masks.to(device).long().squeeze(1)

            outputs = model(pixel_values=images)
            logits  = F.interpolate(outputs.logits, size=masks.shape[-2:],
                                    mode='bilinear', align_corners=False)
            preds = logits.argmax(dim=1)
            for p, m in zip(preds, masks):
                val_ious.append(compute_iou(p, m))

    mean_iou = np.mean(val_ious)
    print(f"Epoch {epoch+1:02d} | Loss: {np.mean(train_losses):.4f} | Val IoU: {mean_iou:.4f}")

    # Save best model
    if mean_iou > best_val_iou:
        best_val_iou = mean_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Saved best model (IoU: {mean_iou:.4f})")

Epoch 01 | Loss: 0.4851 | Val IoU: 0.6647
Saved best model (IoU: 0.6647)
Epoch 02 | Loss: 0.4048 | Val IoU: 0.6708
Saved best model (IoU: 0.6708)
Epoch 03 | Loss: 0.3749 | Val IoU: 0.6913
Saved best model (IoU: 0.6913)
Epoch 04 | Loss: 0.3672 | Val IoU: 0.7037
Saved best model (IoU: 0.7037)
Epoch 05 | Loss: 0.3630 | Val IoU: 0.7122
Saved best model (IoU: 0.7122)
Epoch 06 | Loss: 0.3332 | Val IoU: 0.7042
Epoch 07 | Loss: 0.3469 | Val IoU: 0.7029
Epoch 08 | Loss: 0.3269 | Val IoU: 0.7178
Saved best model (IoU: 0.7178)
Epoch 09 | Loss: 0.3357 | Val IoU: 0.7175
Epoch 10 | Loss: 0.3130 | Val IoU: 0.7190
Saved best model (IoU: 0.7190)
Epoch 11 | Loss: 0.3075 | Val IoU: 0.7211
Saved best model (IoU: 0.7211)
Epoch 12 | Loss: 0.3107 | Val IoU: 0.7170
Epoch 13 | Loss: 0.3092 | Val IoU: 0.7285
Saved best model (IoU: 0.7285)
Epoch 14 | Loss: 0.3169 | Val IoU: 0.7258
Epoch 15 | Loss: 0.3180 | Val IoU: 0.7212
Epoch 16 | Loss: 0.3075 | Val IoU: 0.7297
Saved best model (IoU: 0.7297)
Epoch 17 | Loss: 0

In [16]:
print(best_val_iou)

0.7435310954848925


In [17]:
all_preds = []
all_masks = []

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device).long().squeeze(1)

        outputs = model(pixel_values=images)
        logits =  F.interpolate(outputs.logits, size = masks.shape[-2:],
                                mode = 'bilinear', align_corners=False)
        
        preds = logits.argmax(dim=1)

        all_preds.append(preds.cpu().numpy().flatten())
        all_masks.append(masks.cpu().numpy().flatten())

all_preds = np.concatenate(all_preds)
all_masks = np.concatenate(all_masks)

accuracy  = accuracy_score(all_masks,  all_preds)
precision = precision_score(all_masks, all_preds)
recall    = recall_score(all_masks,    all_preds)
f1        = f1_score(all_masks,        all_preds)

print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")

# Full breakdown by class
print("\n--- Classification Report ---")
print(classification_report(all_masks, all_preds,
                             target_names=['Background', 'Plant']))
    

Accuracy:  0.9336 (93.36%)
Precision: 0.8640 (86.40%)
Recall:    0.9466 (94.66%)
F1 Score:  0.9035 (90.35%)

--- Classification Report ---
              precision    recall  f1-score   support

  Background       0.97      0.93      0.95   3235696
       Plant       0.86      0.95      0.90   1581200

    accuracy                           0.93   4816896
   macro avg       0.92      0.94      0.93   4816896
weighted avg       0.94      0.93      0.93   4816896

